# Notebook 01: Formulacion del problema, hipotesis y datos

## 1.1 Descripcion del problema real

**El contexto.** Las plataformas de reparto a domicilio le prometen al cliente una hora
estimada de llegada en el momento en que confirma el pedido. Esa promesa condiciona tres
decisiones a la vez: si el cliente compra o abandona el carrito, a que repartidor se le
asigna el pedido, y cuando conviene que la cocina empiece a preparar. Una estimacion
equivocada por exceso pierde la venta antes de que exista; una equivocada por defecto deja
al cliente esperando con la comida enfriandose.

**Las partes involucradas.**

| Parte | Que le importa el tiempo estimado |
|---|---|
| El cliente | Decide si compra. Un tiempo largo lo manda a la competencia y uno incumplido hace que no vuelva |
| El repartidor | Su carga de trabajo y sus incentivos dependen de cuantos pedidos entra por turno |
| El restaurante | Ajusta el momento de coccion para que la comida salga justo cuando el repartidor llega |
| La plataforma | Asigna repartidores, define zonas de cobertura y responde por los reclamos |

**Por que es un problema de regresion.** Lo que interesa predecir es el tiempo transcurrido
entre que se toma el pedido y se entrega, medido en minutos. Es una magnitud continua sobre
una escala con unidad conocida, no una etiqueta. La pregunta no es *si* el pedido llegara
tarde sino *cuantos minutos* tardara, porque el numero es lo que se le muestra al cliente y
lo que alimenta la asignacion de repartidores.

**La relevancia para Santa Cruz de la Sierra.** El mercado local de reparto por aplicacion
esta dominado por operadores como PedidosYa, que trabajan sobre una ciudad de anillos
concentricos, con congestion marcada en horario pico y una temporada de lluvias que cambia
las condiciones de circulacion durante varios meses al ano. Son exactamente las condiciones
que este experimento pone a prueba.

## 1.2 Identificacion de variables

**Variable dependiente.** `minutos`, el tiempo total de entrega. Continua, en minutos,
observada al cerrar el pedido.

**Variables independientes.** Se seleccionaron once, todas conocidas o estimables en el
momento en que se asigna el pedido. Ese es el filtro que las hace utiles: una variable que
solo se conoce despues de la entrega no sirve para prometerle un tiempo al cliente.

| Variable | Tipo | Escala | Significado y por que se incluye |
|---|---|---|---|
| `distancia_km` | Numerica | Continua, km | Distancia en linea recta entre restaurante y destino. Es la variable que la intuicion senala primero y la que la hipotesis pone a prueba |
| `edad` | Numerica | Discreta, anos | Edad del repartidor. Aproxima experiencia y estilo de conduccion |
| `calificacion` | Numerica | Continua, 1 a 5 | Calificacion historica del repartidor. Resume su desempeno pasado en un solo numero |
| `pedidos_simultaneos` | Numerica | Discreta, conteo | Cuantos pedidos lleva el repartidor a la vez. Cada pedido extra agrega una parada al recorrido |
| `estado_vehiculo` | Numerica | Ordinal, 0 a 2 | Condicion del vehiculo. Uno en mal estado circula mas lento |
| `trafico` | Categorica | Ordinal: bajo, medio, alto, atasco | Densidad de transito al momento del pedido |
| `clima` | Categorica | Nominal, 6 estados | Condicion meteorologica. Afecta velocidad y seguridad de circulacion |
| `vehiculo` | Categorica | Nominal, 3 tipos | Motocicleta, scooter o scooter electrico |
| `tipo_pedido` | Categorica | Nominal, 4 tipos | Bebidas, comida, buffet o snack. Aproxima el tiempo de preparacion |
| `ciudad` | Categorica | Nominal, 3 tipos | Urbana, metropolitana o semiurbana. Aproxima densidad y calidad vial |
| `festivo` | Categorica | Binaria | Si el pedido cae en dia festivo. Cambia el volumen de demanda y de transito |

**Que se descarto y por que.** Se excluyen los identificadores (`ID`, `Delivery_person_ID`)
porque son etiquetas arbitrarias sin orden ni magnitud, y las cuatro columnas de latitud y
longitud crudas, que se reemplazan por la distancia derivada de ellas. Conservar las
coordenadas ademas de la distancia agregaria colinealidad sin aportar informacion nueva.

## 2.1 Planteamiento de la hipotesis

> **La distancia entre el restaurante y el destino predice positivamente el tiempo de
> entrega, pero la densidad de trafico explica una porcion mayor de su variacion que la
> distancia misma.**

La hipotesis es doble a proposito, y cada mitad se verifica por separado con un criterio
numerico distinto:

**Primera parte, la direccion.** El coeficiente de `distancia_km` en la regresion lineal
multiple debe ser positivo, igual que su correlacion con `minutos`. Se rechaza si el signo
sale negativo o si la relacion no se distingue de cero.

**Segunda parte, la magnitud relativa.** La proporcion de varianza del tiempo explicada por
`trafico` debe superar a la explicada por `distancia_km`. Se mide calculando el coeficiente
de determinacion de cada variable por separado. Se rechaza si la distancia explica mas.

**Bajo que condiciones.** La afirmacion vale para pedidos urbanos de comida preparada, en
distancias de hasta veinte kilometros, con el trafico observado al momento de la asignacion.
Fuera de ese rango no se sostiene: en un reparto interurbano de cien kilometros la distancia
volveria a dominar porque el trayecto deja de transcurrir en zona congestionada.

**Por que importa que la segunda parte sea falsable.** Si resultara falsa, la plataforma
puede seguir estimando el tiempo con la distancia, que es el dato mas barato de obtener. Si
resulta verdadera, el sistema necesita alimentarse de trafico en tiempo real, y eso cambia
la arquitectura del producto y su costo operativo.

## 2.2 Justificacion de la hipotesis

**Argumento tecnico.** El tiempo de un trayecto es la distancia dividida por la velocidad
media. La distancia esta en el numerador, de modo que su efecto es directo y su signo
positivo queda garantizado por construccion. La velocidad, en cambio, esta en el
denominador, y es justamente lo que la congestion degrada. En transito libre un repartidor
urbano sostiene entre 25 y 35 km/h; dentro de un atasco esa velocidad cae a valores de un
solo digito.

De ahi sale la segunda parte de la hipotesis. El rango de distancias de un reparto urbano es
estrecho, unos pocos kilometros, mientras que el rango de velocidades posibles abarca un
orden de magnitud completo. Cuando un cociente tiene un numerador que varia poco y un
denominador que varia mucho, la variacion del resultado queda dominada por el denominador.

**Evidencia bibliografica.** La literatura sobre estimacion de tiempos de reparto coincide
en que las variables de contexto pesan mas que la geometria del recorrido:

- **Sharma, K., et al. (2025).** *Food Delivery Time Prediction in Indian Cities Using
  Machine Learning Models.* arXiv:2503.15177. Trabaja sobre este mismo conjunto de datos y
  concluye que incorporar trafico y clima es lo que produce la mejora sustantiva sobre
  modelos basados solo en geometria.
- **Yildiz, B., y Cagdas, G. (2024).** *A Comparative Analysis of Machine Learning Models
  for Time Prediction in Food Delivery Operations.* Artificial Intelligence Theory and
  Applications, 4(1). Reporta que la intensidad del trafico y el perfil del repartidor
  aparecen sistematicamente entre los predictores mas informativos.
- **Zychowski, A., y Mandziuk, J. (2024).** *Bayesian Modeling of Travel Times on the
  Example of Food Delivery.* Electronics, 13(17), 3418. Documenta que las plataformas suelen
  carecer de datos de trafico en tiempo real, y que esa ausencia es la principal fuente de
  error en sus estimaciones.

Las tres apuntan al mismo lugar: la distancia es necesaria pero insuficiente, y el contexto
de circulacion es lo que separa una estimacion util de una inservible.

## 3.1 Seleccion y descripcion del conjunto de datos

**Fuente.** Conjunto *Food Delivery Time Prediction*, distribuido publicamente en el
repositorio `Vikranth3140/Food-Delivery-Time-Prediction` a partir de registros operativos de
reparto a domicilio en ciudades de India. Se lee desde una URL publica: no requiere descarga
manual ni credenciales, de modo que cualquiera puede reejecutar el experimento desde cero.

**Formato.** Archivo CSV plano, codificacion UTF-8, separador coma, aproximadamente 7 MB.

**Volumen.** 45.593 pedidos y 20 columnas.

**Contenido.** Cada fila es un pedido entregado. Registra el identificador del repartidor y
su perfil, las coordenadas del restaurante y del destino, la fecha y la hora del pedido y de
la recogida, las condiciones de clima y trafico al momento de la entrega, el tipo de pedido
y de vehiculo, la ciudad, si era dia festivo, y el tiempo total en minutos.

**Por que es idoneo para este problema.** Reune las cuatro condiciones que la hipotesis
necesita para ser contrastable:

1. **La variable objetivo es continua y esta en unidades interpretables.** Minutos, no una
   categoria de rapidez. Eso permite que las metricas de error se lean directamente como
   minutos de equivocacion.
2. **Trae la geometria y el contexto por separado.** Las coordenadas permiten construir la
   distancia y las columnas de trafico y clima permiten medir el contexto. Sin ambas cosas
   la segunda parte de la hipotesis no se podria poner a prueba.
3. **Tiene variables categoricas reales.** Seis, con cardinalidades entre dos y siete, que
   obligan a resolver la codificacion en lugar de saltearla.
4. **El volumen alcanza para una particion honesta.** Con 45.593 filas, un conjunto de
   prueba del 20 % deja mas de ocho mil observaciones, suficiente para que las metricas sean
   estables y no dependan de la suerte del sorteo.

**Limitaciones que se declaran de entrada.** Los registros provienen de ciudades de India,
no de Bolivia. La estructura del problema es la misma, pero las magnitudes concretas
(velocidades medias, distancias tipicas) no se trasladan sin recalibrar. El conjunto tampoco
registra el tiempo de preparacion en cocina, que es una fuente real de variacion que ninguna
de las variables disponibles captura, y esa ausencia va a aparecer despues como un techo en
el desempeno alcanzable.

### Entorno y persistencia de resultados

La celda siguiente fija tres cosas de las que depende que el experimento sea reproducible:

**Semilla fija.** `RANDOM_STATE = 42` se aplica a la particion train/test y al ajuste de
todos los modelos. Sin ella cada ejecucion produciria particiones distintas y las metricas
no serian comparables entre corridas ni verificables por un tercero.

**Persistencia en Google Drive.** Los resultados se escriben en `MyDrive/food_delivery` y no
en el disco temporal de Colab, que se borra al cerrar la sesion. Eso permite ejecutar el
experimento en varias sesiones sin repetir etapas: el notebook 02 deja las particiones
preparadas y los notebooks 03, 04 y 05 las consumen tal cual, de modo que todos operan sobre
exactamente los mismos datos. Si el montaje no se completa, la celda interrumpe la ejecucion
en lugar de escribir en una ubicacion volatil, porque un fallo silencioso ahi haria que el
notebook 04 no encuentre lo que el 02 dejo.

**Registro de figuras.** La funcion `guardar` escribe cada grafico en `splits/` dentro de la
carpeta del proyecto. Todas las figuras terminan en el mismo lugar, listas para el informe.

In [ ]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)
sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 110
RANDOM_STATE = 42

PROYECTO = "food_delivery"

EN_DRIVE = False
try:
    from google.colab import drive
    drive.mount("/content/drive")
    if not os.path.isdir("/content/drive/MyDrive"):
        raise RuntimeError(
            "Drive no quedo montado. Volve a ejecutar esta celda y autoriza el acceso "
            "en la ventana emergente."
        )
    RUTA = f"/content/drive/MyDrive/{PROYECTO}"
    EN_DRIVE = True
except ImportError:
    RUTA = os.path.abspath(f"./{PROYECTO}")

CARPETA_SPLITS = os.path.join(RUTA, "splits")
os.makedirs(CARPETA_SPLITS, exist_ok=True)


def guardar(nombre):
    destino = os.path.join(CARPETA_SPLITS, nombre + ".png")
    plt.savefig(destino, dpi=150, bbox_inches="tight")
    print("Figura guardada:", destino)


print("Persistencia en Drive:", EN_DRIVE)
print("Ruta de trabajo:", RUTA)

### Carga del conjunto

Se lee directamente desde la URL publica. La primera lectura tarda unos segundos.

In [ ]:
URL = "https://raw.githubusercontent.com/Vikranth3140/Food-Delivery-Time-Prediction/main/datasets/kaggle/train.csv"

crudo = pd.read_csv(URL)
crudo.columns = [c.strip() for c in crudo.columns]

print("Dimensiones:", crudo.shape)
crudo.head()

### Inspeccion inicial

Antes de decidir cualquier tratamiento hay que ver que trae realmente el archivo.

In [ ]:
crudo.info()

**Lectura de la inspeccion.** Casi todas las columnas llegan como texto, incluidas las que
deberian ser numericas. `Time_taken(min)` viene con el prefijo `"(min) "` pegado al numero y
`Weatherconditions` con el prefijo `"conditions "`. Ademas los valores de texto arrastran
espacios sobrantes al final. Son problemas de formato, no de contenido, y se resuelven en el
notebook 02.

In [ ]:
print("Nulos que pandas detecta al leer el archivo:", int(crudo.isna().sum().sum()))
print("Filas duplicadas:", int(crudo.duplicated().sum()))

**Cero nulos, y ahi esta la trampa.** El archivo no tiene ninguna celda vacia, pero eso no
significa que la informacion este completa. Las ausencias vienen escritas como el texto
`"NaN "`, con un espacio al final que impide que pandas las reconozca como valor faltante.
Para pandas son cadenas validas, igual que `"Urban"` o `"Sunny"`.

Si no se corrigiera, cada una de esas cadenas terminaria convertida en una categoria mas por
el codificador, y el modelo aprenderia a tratar "no sabemos" como si fuera un estado del
mundo. La celda siguiente las destapa quitando los espacios y convirtiendolas en nulos reales.

In [ ]:
destapado = crudo.copy()
for col in destapado.select_dtypes(include=["object", "string"]).columns:
    destapado[col] = destapado[col].astype(str).str.strip()
destapado = destapado.replace(["NaN", "nan", ""], np.nan)

faltantes = destapado.isna().sum()
faltantes = faltantes[faltantes > 0].sort_values(ascending=False)

resumen = pd.DataFrame({
    "nulos": faltantes,
    "porcentaje": (100 * faltantes / len(destapado)).round(2),
})
print("Nulos reales una vez destapados:", int(faltantes.sum()))
resumen

**Lectura de los nulos reales.** Aparecen siete columnas con ausencias y ninguna supera el
4 % de las filas. Son pocas y estan repartidas entre columnas distintas, lo que permite
tratarlas sin comprometer el volumen del conjunto. El notebook 02 decide que hacer con cada
grupo.

Notar que el clima trae la categoria `"conditions NaN"`: ahi el prefijo esta pegado al
marcador de ausencia, de modo que hay que quitar el prefijo antes de poder destaparla. Es la
razon por la que en el notebook 02 la limpieza de texto va primero y el tratamiento de nulos
despues.

In [ ]:
print("Categorias declaradas en cada variable categorica:")
for c in ["Weatherconditions", "Road_traffic_density", "Type_of_order",
          "Type_of_vehicle", "City", "Festival"]:
    valores = sorted(crudo[c].str.strip().dropna().unique())
    print(f"  {c:24s} {len(valores):2d}  {valores}")